In [25]:
!wget -q https://github.com/ping-long-github/architecture-explorer/blob/main/system_context.md -O systemcontext.md
!pip install -q google-generativeai gradio

In [ ]:
import google.generativeai as genai
import gradio as gr
import os

import google.generativeai as genai

# # This prompts the user to grant access to their Colab secrets
# try:
#     GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
#     genai.configure(api_key=GOOGLE_API_KEY)
# except userdata.SecretNotFoundError:
#     print("Please set your GOOGLE_API_KEY in the Colab Secrets (key icon on the left sidebar).")

# # 1. Set your Free API Key
# genai.configure(api_key=GOOGLE_API_KEY)

import os
import google.generativeai as genai

def setup_gemini_api():
    api_key = None

    # 1. Try Environment Variables (Local VS Code, standard Jupyter, Hugging Face)
    # Try to load a .env file if the user has python-dotenv installed
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass 
    
    api_key = os.environ.get("GOOGLE_API_KEY")
    if api_key:
        print("Environment detected: Local/Server. Key loaded from Environment Variables.")

    # 2. Try Kaggle Secrets (if applicable)
    if not api_key:
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret("GOOGLE_API_KEY")
            print("Environment detected: Kaggle. Key loaded from User Secrets.")
        except ImportError:
            pass # Not running in Kaggle

    # 3. Try Google Colab Secrets (Moved to the last automated step)
    if not api_key:
        try:
            from google.colab import userdata
            api_key = userdata.get('GOOGLE_API_KEY')
            print("Environment detected: Google Colab. Key loaded from Secrets.")
        except ImportError:
            pass # Not running in Google Colab

    # 4. Fallback: Prompt the user manually if no key is found automatically
    if not api_key:
        import getpass
        print("No API key found in environment or secrets.")
        api_key = getpass.getpass("Please paste your Gemini API Key here: ")

    # Final Check and Configuration
    if api_key:
        genai.configure(api_key=api_key)
        print("Gemini API configured successfully!")
    else:
        raise ValueError("Failed to configure Gemini API. No key provided.")

# Run the setup
setup_gemini_api()

# 2. Read the System Prompt from the external file
# Ensure 'systemcontext.md' is uploaded to your Colab environment or repository
try:
    with open("systemcontext.md", "r") as file:
        SYSTEM_PROMPT = file.read()
except FileNotFoundError:
    SYSTEM_PROMPT = "You are an elite AI Compute Performance Engineer..."
    print("Warning: systemcontext.md not found. Using fallback prompt.")

# 3. Initialize the Model with the dynamically loaded prompt
model = genai.GenerativeModel(
    model_name="gemini-1.5-flash",
    system_instruction=SYSTEM_PROMPT
)

# 4. Define the Chat Logic
# 5. Define the Chat Logic (Patched for empty messages)
def chat_with_companion(message, history):
    # SAFETY CHECK 1: Prevent sending empty messages to the Gemini API
    if not message or not message.strip():
        return "Please enter a valid question or code snippet."
    
    formatted_history = []
    for user_msg, ai_msg in history:
        # SAFETY CHECK 2: Prevent empty history payloads
        if user_msg and ai_msg:
            formatted_history.append({"role": "user", "parts": [user_msg]})
            formatted_history.append({"role": "model", "parts": [ai_msg]})
    
    # Start chat and send the safe message
    chat = model.start_chat(history=formatted_history)
    response = chat.send_message(message)
    return response.text

# 5. Launch the Web UI
demo = gr.ChatInterface(
    fn=chat_with_companion,
    title="Architecture Explorer: AI Compute Companion",
    description="Paste your kernel code (C, Rust, Mojo, Triton). I will analyze Cache Lines, Row/Col Major layouts, L1-L3 hits, and Arithmetic Intensity across GPUs and LPUs!",
    theme="glass"
)

# Setting share=True generates a public link
demo.launch(share=True)